# jF_01 - Uji pendahuluan: apakah MLP kita kurang terlatih? (JURNAL)

**Latar:** pada 9 fitur yang sama, pohon gradient-boosting per client mencapai MCC 0,84-0,95 di domainnya sendiri,
sedangkan MLP local-only kita hanya 0,59-0,65. Reviewer akan bertanya apakah angka MCC kita rendah karena modelnya lemah.

**Uji ini TIDAK memakai federasi.** Satu MLP dilatih di satu client dan diuji di test client itu sendiri (dalam-domain),
dengan lima pengaturan yang makin kuat, lalu dibandingkan dengan pohon. Satu seed (42) cukup untuk uji pendahuluan
(simpangan baku local-only sebelumnya hanya +/-0,005).

| Pengaturan | Preproses | Optimizer | Epoch | Arsitektur |
|---|---|---|---|---|
| S0 dasar (replika setup kita) | z-score | SGD lr 3e-3 | 100 | 128-64 |
| S1 anggaran | z-score | SGD lr 3e-3 | 300 | 128-64 |
| S2 optimizer | z-score | Adam lr 1e-3 | 30 | 128-64 |
| S3 + transformasi | log1p bertanda + z-score | Adam lr 1e-3 | 30 | 128-64 |
| S4 + lebih dalam | log1p bertanda + z-score | Adam lr 1e-3 | 30 | 256-128-64 |

**Uji kewajaran (S0):** hasil S0 pada near@9 harus dalam +/-0,03 dari hasil local-only sebelumnya (cic2017 0,645; cic2018 0,592).

**Kriteria dibaca (disepakati SEBELUM melihat angka)**, per client: `gain` = MCC terbaik S1-S4 dikurangi MCC S0; `gap` = MCC pohon dikurangi MCC S0.
- **"MLP kurang terlatih"** bila `gain >= 0,5 x gap` **dan** `gain >= 0,10` pada minimal 3 dari 4 client.
- **"Bukan karena anggaran/optimizer/preproses"** bila `gain < 0,05` pada semua client.
- Selain itu: **sebagian**; laporkan apa adanya.
Selain MCC (ambang 0,5) dilaporkan juga AUC, supaya terlihat apakah masalahnya peringkat atau ambang keputusan.

**Prasyarat:** data far-domain (s01) dan near-domain (`jA_01`) sudah di S3. **Output:** `journal_out/probe_mlp.csv` dan S3 `federated_journal/probe/results/`.
Jalankan di SageMaker.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','boto3','scipy','pandas','scikit-learn'):
    mod='sklearn' if m=='scikit-learn' else m
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, json, time, numpy as np, pandas as pd, boto3
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, matthews_corrcoef

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
FAR_ROOT='unsw-far/federated_survey'; NEAR_ROOT='federated_journal/neardomain'
OUT='federated_journal/probe'
OUTDIR='journal_out'; os.makedirs(OUTDIR,exist_ok=True)
s3=boto3.client('s3',region_name=REGION)
NINE=['flow duration','total fwd packets','total backward packets',
      'total length of fwd packets','total length of bwd packets',
      'fwd packet length mean','bwd packet length mean','flow bytes/s','bwd packets/s']
def npz(key): return np.load(io.BytesIO(s3.get_object(Bucket=BUCKET,Key=key)['Body'].read()),allow_pickle=True)
def load(root,c):
    D={}
    for sp in ('train','test'):
        d=npz(f'{root}/data/{c}/{c}_{sp}.npz'); D[sp]=(np.asarray(d['X'],float),np.asarray(d['y'],int)); D['feats']=[str(f) for f in d['feats']]
    return D
DATA={}
for c in ('cic','unsw'): DATA[('far@9',c)]=load(FAR_ROOT,c)
for c in ('cic2017','cic2018'):
    D=load(NEAR_ROOT,c); idx=[D['feats'].index(f) for f in NINE]
    DATA[('near@9',c)]={sp:(D[sp][0][:,idx],D[sp][1]) for sp in ('train','test')}
for k,v in DATA.items(): print(k,'train',v['train'][0].shape,'pos=%.3f'%v['train'][1].mean(),'| test',v['test'][0].shape)

In [ ]:
def prep(Xtr,Xte,mode):
    f=(lambda X: np.sign(X)*np.log1p(np.abs(X))) if mode=='log' else (lambda X: X)
    A=f(Xtr); B=f(Xte); mu=A.mean(0); sd=A.std(0); sd[sd==0]=1.0
    return (A-mu)/sd,(B-mu)/sd

def init_w(layers,seed):
    rng=np.random.RandomState(seed); W=[]
    for a,b in zip(layers[:-1],layers[1:]): W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def fwd_all(W,X):
    acts=[X]; h=X; L=len(W)//2
    for i in range(L):
        z=h@W[2*i]+W[2*i+1]
        h=1.0/(1.0+np.exp(-np.clip(z,-30,30))) if i==L-1 else np.maximum(0,z); acts.append(h)
    return acts
def predict(W,X): return fwd_all(W,X)[-1].ravel()

def train(Xtr,ytr,hid,opt,lr,epochs,seed=42,batch=256):
    d=Xtr.shape[1]; W=init_w([d]+hid+[1],seed); L=len(W)//2
    rng=np.random.RandomState(seed); n=len(ytr)
    if opt=='adam': m=[np.zeros_like(w) for w in W]; v=[np.zeros_like(w) for w in W]; t=0
    for ep in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=Xtr[bi]; yb=ytr[bi].astype(float)
            acts=fwd_all(W,Xb); o=acts[-1].ravel()
            delta=((o-yb)/len(yb))[:,None]; grads=[None]*(2*L)
            for i in range(L-1,-1,-1):
                grads[2*i]=acts[i].T@delta; grads[2*i+1]=delta.sum(0)
                if i>0: delta=(delta@W[2*i].T)*(acts[i]>0)
            if opt=='sgd':
                for i in range(2*L): W[i]=W[i]-lr*grads[i]
            else:
                t+=1
                for i in range(2*L):
                    m[i]=0.9*m[i]+0.1*grads[i]; v[i]=0.999*v[i]+0.001*grads[i]**2
                    W[i]=W[i]-lr*(m[i]/(1-0.9**t))/(np.sqrt(v[i]/(1-0.999**t))+1e-8)
    return W

def evaluate(p,y):
    return float(matthews_corrcoef(y,(p>=0.5).astype(int))),float(roc_auc_score(y,p))
print('MLP generik siap')

In [ ]:
SETTINGS={
 'S0 dasar (SGD, 100 ep)':        dict(pre='z',  hid=[128,64],     opt='sgd',  lr=3e-3, epochs=100),
 'S1 anggaran (SGD, 300 ep)':     dict(pre='z',  hid=[128,64],     opt='sgd',  lr=3e-3, epochs=300),
 'S2 optimizer (Adam, 30 ep)':    dict(pre='z',  hid=[128,64],     opt='adam', lr=1e-3, epochs=30),
 'S3 + log1p (Adam, 30 ep)':      dict(pre='log',hid=[128,64],     opt='adam', lr=1e-3, epochs=30),
 'S4 + lebih dalam (Adam, 30 ep)':dict(pre='log',hid=[256,128,64], opt='adam', lr=1e-3, epochs=30),
}
rows=[]
for key,D in DATA.items():
    Xtr,ytr=D['train']; Xte,yte=D['test']
    for name,cfg in SETTINGS.items():
        t0=time.time(); A,B=prep(Xtr,Xte,cfg['pre'])
        W=train(A,ytr,cfg['hid'],cfg['opt'],cfg['lr'],cfg['epochs'])
        m,a=evaluate(predict(W,B),yte)
        rows.append(dict(config=key[0],client=key[1],setting=name,mcc=round(m,4),auc=round(a,4)))
        print(f'{key[0]:7s} {key[1]:8s} {name:32s} MCC={m:+.3f} AUC={a:.3f}  [{time.time()-t0:.0f}s]')
    t0=time.time(); rs=np.random.RandomState(0); i=rs.choice(len(Xtr),size=min(100000,len(Xtr)),replace=False)
    tree=HistGradientBoostingClassifier(max_iter=100,random_state=0).fit(Xtr[i],ytr[i])
    m,a=evaluate(tree.predict_proba(Xte)[:,1],yte)
    rows.append(dict(config=key[0],client=key[1],setting='POHON (HistGradientBoosting)',mcc=round(m,4),auc=round(a,4)))
    print(f'{key[0]:7s} {key[1]:8s} {"POHON (HistGradientBoosting)":32s} MCC={m:+.3f} AUC={a:.3f}  [{time.time()-t0:.0f}s]\n')
df=pd.DataFrame(rows); df.to_csv(os.path.join(OUTDIR,'probe_mlp.csv'),index=False)
s3.upload_file(os.path.join(OUTDIR,'probe_mlp.csv'),BUCKET,f'{OUT}/results/probe_mlp.csv')
print('CSV tersimpan + upload')

In [ ]:
print('=== MCC per client dan pengaturan ===')
print(df.pivot_table(index=['config','client'],columns='setting',values='mcc').round(3).to_string())
print('\n=== AUC per client dan pengaturan ===')
print(df.pivot_table(index=['config','client'],columns='setting',values='auc').round(3).to_string())

REF={'cic2017':0.6453,'cic2018':0.5918}   # local-only dalam-domain, jC_01 (near@9)
s0n='S0 dasar (SGD, 100 ep)'
print('\nUji kewajaran S0 vs hasil local-only sebelumnya (near@9):')
for c,ref in REF.items():
    v=float(df[(df.client==c)&(df.config=='near@9')&(df.setting==s0n)].mcc.iloc[0])
    print(f'  {c}: S0={v:.3f} vs {ref:.3f}  |selisih|={abs(v-ref):.3f} <= 0.03 ? {abs(v-ref)<=0.03}')

res=[]
for (cfg,c),g in df.groupby(['config','client']):
    g=g.set_index('setting').mcc; s0=float(g[s0n]); tree=float(g['POHON (HistGradientBoosting)'])
    best=float(max(g[k] for k in SETTINGS if k!=s0n)); gain=best-s0; gap=tree-s0
    res.append(dict(config=cfg,client=c,S0=s0,best_S1_S4=best,tree=tree,gain=gain,gap=gap,
                    trained=(gain>=0.5*gap and gain>=0.10),flat=(gain<0.05)))
R=pd.DataFrame(res); print('\n=== gain vs gap ===\n',R.round(3).to_string(index=False))
n_tr=int(R.trained.sum()); n_flat=int(R.flat.sum())
print(f'\nKriteria (disepakati sebelum melihat angka):')
print(f'  "MLP kurang terlatih"  (gain>=0.5*gap dan gain>=0.10 pada >=3 dari 4 client): {n_tr} dari 4 -> {n_tr>=3}')
print(f'  "Bukan karena anggaran/optimizer/preproses" (gain<0.05 pada semua client): {n_flat} dari 4 -> {n_flat==4}')
R.to_csv(os.path.join(OUTDIR,'probe_gain_gap.csv'),index=False)
s3.upload_file(os.path.join(OUTDIR,'probe_gain_gap.csv'),BUCKET,f'{OUT}/results/probe_gain_gap.csv')
print('Jangan menyimpulkan sebelum Kiro membaca hasilnya.')

In [ ]:
NEED=[f'{OUT}/results/probe_mlp.csv',f'{OUT}/results/probe_gain_gap.csv']
resp=s3.list_objects_v2(Bucket=BUCKET,Prefix=f'{OUT}/')
keys=sorted(o['Key'] for o in resp.get('Contents',[]))
print('File hasil di S3 (%d):'%len(keys))
for k in keys: print('  s3://%s/%s'%(BUCKET,k))
miss=[k for k in NEED if k not in keys]
print('LENGKAP' if not miss else 'KURANG: %s'%miss)
print()
print('Di komputer lokal (PowerShell), tarik dengan:')
print('  aws s3 sync s3://%s/%s/ ./probe'%(BUCKET,OUT))